# Benchmarking wildfire spread models

**SCIL, AI Collaborative for Wildfire, Workstream 2.** Red Sky demo, November 2026.

This notebook is the whole interface to the benchmarking framework. Two kinds of file go in:

* a **workflow**: a recipe for predicting a fire (which data, which model, which settings), with no fire in it;
* **cases**: fires, each with its facts (when it started, where, its ignition points, its observed perimeter).

Out comes the workflow's **table**: one row per fire, scored against what the fire really did, with a record in Dagster of how every number was produced. Different recipes are different workflows, so comparing models or settings means comparing their tables.

The example fires are the **Hughes Fire** (January 22, 2025, Lake Hughes Road near Castaic Lake, Los Angeles County, 10,425 acres at containment) and the **Sandia Fire** (October 7, 2025, San Diego County, about 30 acres). Each is scored against a FIRIS perimeter about two hours after discovery, at the time that perimeter was observed. The recipe runs ELMFIRE on real LANDFIRE fuels and real station wind.

| Section | What happens |
|---|---|
| 1. The recipe | The workflow file |
| 2. The fires | The case files |
| 3. From recipe to plan | The resolver checks the recipe against the block registry and plans it for each fire |
| 4. The graph | The plan as the asset graph Dagster will run |
| 5. Identity and reuse | Why the same LANDFIRE is never downloaded twice |
| 6. Compile and push | The plan becomes one file on the WSTC Dagster |
| 7. Run it on the fires | One Dagster run per fire |
| 8. The table | Scores, previews, and where to look in Dagster |
| 9. A second recipe | A different ignition source: what is shared, and a fire it can't run on |
| 10. What is real so far | An honest list |

## 0. Setup

The notebook runs anywhere that can reach the WSTC Dagster over HTTPS: a laptop, or NDP JupyterHub. It needs this repository (block registry, resolver, compiler) plus `pyyaml` and `requests`.

Two things are typed in, not stored: the WSTC dashboard password, and an email address for LANDFIRE's download service (it requires one on every request; it travels as a run tag).

In [ ]:
import sys, getpass, time
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(REPO / "src"), str(REPO / "deploy" / "wstc")]

from bench import wstc
from bench.workflows import load_workflow, load_cases
from bench.notebook import (show_file, plan_tables, mermaid, identity_table, shared_with, md_table,
                            run_fires, asset_rows, show_previews)
from dagster_client import Dagster
from IPython.display import display, Markdown

ME = "apearce"                       # everything we put on the shared server starts with this
WORKFLOW = REPO / "workflows" / "elmfire_reported_ignition.yaml"
FIRES = ["hughes", "sandia"]         # cases/<name>.yaml; None = every case
IMAGES = wstc.image_refs("a3pearce") # the blocks' images, one per family (section 6)
BUILD = REPO / "deploy" / "wstc" / "build"     # compiled files land here (not committed)
BUILD.mkdir(exist_ok=True)

In [ ]:
d = Dagster(password=getpass.getpass("WSTC Dagster password: "))
LFPS_EMAIL = input("Email for LANDFIRE requests: ").strip()
print(len(d.jobs()), "jobs on the server")

## 1. The recipe

A workflow is configuration only. Each **input slot** (landscape, weather, moisture, ignition) has exactly one **source**, optionally followed by **transforms**; then one **model** with fixed parameters, a **reference** and the **metrics**. Every name is a **block**: a manifest in `blocks/` declaring what it consumes and produces and which parameters it takes.

There is no fire in it. The ignition source says "the case's *reported* ignition", not a coordinate.

In [ ]:
show_file(WORKFLOW)

## 2. The fires

A case is one fire's facts: its clock zero (`t0`), weather window, domain, ignition points and observed perimeter. The framework derives each fire's working grid from it (EPSG:5070 at the recipe's 30 m).

In [ ]:
for name in FIRES or sorted(load_cases()):
    show_file(REPO / "cases" / f"{name}.yaml")

## 3. From recipe to plan

The **resolver** applies the recipe to each fire before anything runs. It checks every block exists, is the right kind for where it is used and has an implementation; checks every parameter's type and merges it with the manifest's defaults; inserts automatic steps where a source's native CRS differs from the fire's (the ignition comes in as lat/lon, so it gets a reproject step); and checks the model's required inputs are all produced. A recipe that can't work is rejected here, not two hours into a run.

What can differ from fire to fire (an ignition source the case doesn't have, no weather station in range) is found when the step runs, and recorded as *unavailable*, with the reason (section 9).

In [ ]:
cp = wstc.compile_plan(load_workflow(WORKFLOW), load_cases(FIRES), ME)
plan_tables(cp)

## 4. The graph

Every asset is partitioned by fire, so running the recipe on one fire is one Dagster run, and Dagster can chart any number (burned area, IoU) across fires. `report` gathers the workflow's table.

In [ ]:
mermaid(cp)

### Conventions at every block boundary

Models disagree about almost everything: which way is north in an array, whether wind direction is where it blows from or to, whether canopy cover is a percent or a fraction. The framework fixes one answer for each (`docs/conventions.md`), makes every block convert in and out of it, and checks every raster a block writes before accepting it.

| Rule | Value |
|---|---|
| Grid | the fire's CRS (EPSG:5070) at the recipe's resolution, **north-up**: row 0 is the northern edge |
| Wind direction | degrees clockwise from **grid** north, the direction the wind blows **from** |
| Units | SI: m, m/s, kg/m³, canopy cover as a fraction |
| Time | minutes since the fire's t0; arrival time NaN = never burned |

Two examples from this run: the station-wind block rotates directions from true north to grid north (by 13.7° here: Castaic is about 23° of longitude west of EPSG:5070's central meridian at 96° W, and this Albers projection turns grid north about 0.6° per degree of longitude; without the rotation a Santa Ana from 60° would be fed to the model as one from 46°), and the ELMFIRE adapter flips every grid on the way in and out, because pyretechnics puts row 0 at the **south** edge.

## 5. Identity and reuse

Every artifact gets an **identity**: a hash of everything that determines its content (the block's code, version, parameters and data selection, the facts of the fire it depends on, and its inputs' identities), and is stored under it, not under the workflow's name. Before a step does any work it checks whether its identity is already built.

So every recipe that uses the same LANDFIRE on the same fire shares one download. LANDFIRE's identity only depends on the fire's grid, not on, say, its ignition time. Change anything that matters (the domain, a parameter, the block's code) and the identity changes: nothing stale is ever served.

In [ ]:
identity_table(cp, (FIRES or sorted(cp["fires"]))[0])

## 6. Compile and push

The WSTC server takes **one Python file per upload**, and its standard image has only a handful of packages. So the workflow is compiled into one file, **`apearce_elmfire_reported_ignition.py`**: the plan for every fire plus the blocks' code.

The blocks' packages come from **container images** on Docker Hub, one per family of blocks with the same needs. Each step asks Dagster to start its pod from its block's image:

| Image | Blocks | On top of the server's Dagster image |
|---|---|---|
| `a3pearce/bench-core` | LANDFIRE, roads, wind, moisture, ignition, reproject, reference, metrics | rasterio |
| `a3pearce/bench-elmfire` | ELMFIRE | its own Python 3.12 with a compiled pyretechnics, NumPy 1.26, matplotlib |

That is the real deployment's shape: a block's dependencies live in its image, and the orchestration never has to know them. Each image is tagged by a hash of its recipe (`images/wstc/<family>/`), so the workflow file always names exactly the images it was built for.

In [ ]:
for family, image in IMAGES.items():
    found = wstc.image_published(image)
    print(image, {True: "is on Docker Hub", False: "is NOT on Docker Hub yet: run scripts/build_wstc_image.sh",
                  None: "(couldn't reach Docker Hub to check)"}[found])
wf_file, cp = wstc.build_workflow_file(WORKFLOW, FIRES, ME, BUILD, IMAGES)
jobs = wstc.job_names(ME, cp["workflow"])
print(f"{wf_file.name}: {wf_file.stat().st_size:,} bytes")
print([j for j in d.upload(str(wf_file)) if j.startswith(f"{ME}_{cp['workflow']}")])

## 7. Run it on the fires

One Dagster run per fire does everything for that fire: inputs, model, scores. Independent steps run in parallel pods. The first time, the waiting is LANDFIRE clipping the layers on its side (about half a minute) and the model. The server allows four runs at once across all users, so we launch at most three.

In [ ]:
runs = run_fires(d, jobs["run"], sorted(cp["fires"]), tags={"bench/lfps_email": LFPS_EMAIL})
md_table([{"fire": k, **v, "url": d.url_for(v["run_id"])} for k, v in runs.items()])

In [ ]:
# what each step did, for the first fire
fire = sorted(runs)[0]
res = d.results(runs[fire]["run_id"])
md_table(asset_rows(res, ["reused", "unavailable", "build_s", "grid", "station", "mean_wind_ms",
                          "cells_changed", "time_utc", "reprojected", "burned_area_ha", "iou", "area_bias"]))

## 8. The table

The `report` asset is the workflow's table: one row per fire. **IoU** is the overlap between predicted and observed burned area (1 is perfect); **area bias** is the predicted area relative to the observed (0 is perfect, -0.5 means half). The same table is written as CSV for comparing workflows offline.

In [ ]:
run_id, status, res = d.run(jobs["report"])
rep = next(md for a, md in res.items() if a.endswith("/report"))
display(md_table(rep["rows"]))
print("CSV:", rep.get("csv_s3") or rep.get("csv"))

Every preview is drawn by the framework, not the model, with the same colours and fixed scales, so previews from different models, recipes and fires compare by eye: time of arrival in hours after t0, and fireline intensity on a log scale.

In [ ]:
previews = {}
for fire, v in runs.items():
    for asset, md in d.results(v["run_id"]).items():
        if asset.endswith("/prediction") and "preview" in md:
            previews[fire] = md["preview"]
show_previews(previews)
print(f"{d.url}/assets/{ME}/{cp['workflow']}/prediction")

### Where to look in Dagster

* **Assets > apearce > elmfire_reported_ignition > prediction**: the partitions are the fires; any numeric field is charted across them.
* Every materialization records its identity, the artifact path, the run that built it, whether it was reused, and the block's full summary.
* **Runs** has every launch with its logs, including each block's own output.

## 9. A second recipe

`elmfire_ngfs_ignition.yaml` is the same recipe with one change: the ignition is the fire's first NGFS (GOES) detection instead of the reported origin. It is a different workflow, because it is a different recipe. Two things to see:

* it **shares** every artifact that doesn't depend on the ignition (LANDFIRE, the roads override, the wind, the moisture), so those are reused, not rebuilt;
* both cases have an NGFS point, so it runs for both. A case without one would be **unavailable** for this recipe, with the reason, instead of failing.

In [ ]:
WORKFLOW_2 = REPO / "workflows" / "elmfire_ngfs_ignition.yaml"
wf_file_2, cp_2 = wstc.build_workflow_file(WORKFLOW_2, FIRES, ME, BUILD, IMAGES)
jobs_2 = wstc.job_names(ME, cp_2["workflow"])
shared_with(cp, cp_2, sorted(cp["fires"])[0])

In [ ]:
d.upload(str(wf_file_2))
runs_2 = run_fires(d, jobs_2["run"], sorted(cp_2["fires"]), tags={"bench/lfps_email": LFPS_EMAIL})
run_id, status, res = d.run(jobs_2["report"])
md_table(next(md for a, md in res.items() if a.endswith("/report"))["rows"])

## 10. What is real so far

| Part | Status |
|---|---|
| LANDFIRE 2024 fuels and canopy, LF 2020 elevation | real, from the LANDFIRE Product Service |
| Roads/urban override to GR2 | real |
| Station wind (nearest ASOS with data, hourly) | real, from the Iowa Environmental Mesonet (the nearest station with data; Sandberg, SDB, for Hughes); uniform over the domain (no WindNinja yet) |
| Fuel moisture | a fixed scenario value (1 h dead = 6%), other classes at the adapter's reported defaults |
| Ignition: reported origin | real (the WIFIRE fire groupings geopackage, Centers layer) |
| Ignition: NGFS first detection | real (the same geopackage, NGFS layer) |
| ELMFIRE (pyretechnics) | real, in its own image |
| FARSITE, WindNinja, WRF-Fire | manifests only: a workflow naming them is rejected by the resolver with the reason |
| Reference perimeter, IoU, area bias | real: FIRIS perimeters, scored at their observation time |

Next: more fires (a case catalog built from the lab's benchmark geopackage), the model duration defaulted from each reference's observation time, and finished artifacts in the S3 bucket.

### Cleanup (optional)

Removes our files from the server; the jobs and assets disappear with them. Artifacts on the shared volume stay, so a later upload reuses them.

In [ ]:
# d.remove(wf_file.name); d.remove(wf_file_2.name)